# Step 2 NIRCam WFSS Reduction + Extraction w/ Grizli

## Run this notebook after notebook 01. This routine extracts spectra for all possible objects provided by an input catalog. Extraction happens simultaneously across all filters, grisms and modules.

In [ ]:
# Import necessary modules
import os, glob, shutil, warnings, time, gc, re, traceback, subprocess, tempfile, hashlib, json
from pathlib import Path
from collections import OrderedDict
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits, ascii
from astropy.table import Table, vstack
import astropy.wcs as pywcs
from astropy.table import Table
import astropy.units as u
from astropy.coordinates import SkyCoord

warnings.filterwarnings('ignore', category=pywcs.FITSFixedWarning)

# Catalog information
TARGET_CATALOG = 'A2744_clean_coordinates.csv'    # any table astropy can read (csv, ecsv, fits, ...), or None
NAME_COL, RA_COL, DEC_COL = 'ID', 'RA', 'DEC'   # column names in TARGET_CATALOG (NAME_COL=None -> numbered). RA/Dec: degrees or sexagesimal

# Define whether a target is matched to the existing segmentation images, or if it is a new source
MATCH_RADIUS_ARCSEC      = 0.5    # a target within this distance of a detected segment uses that source
ADD_UNDETECTED           = True   # targets with no detection get a new small segment at their position
NEW_SOURCE_RADIUS_ARCSEC = 0.2    # radius of that segment (also used for its aperture magnitude)
COARSE_MARGIN_ARCSEC     = 30     # safety margin added to the mosaic radius in the quick pre-selection


# Contamination modelling parameters
PAD           = 800               # padding (px) around the detector for off-chip sources
CPU_COUNT     = 4
MAG_LIMIT_MODEL = 26              # sources brighter than this go into the contamination model
FCONTAM       = 0.2               # contamination weighting used in MultiBeam
CUTOUT_MARGIN = 25                # px added to the largest trace offset to size object cutouts / beams
USE_JWST_CRDS = False             # False: use the GRISM_NIRCAM config files downloaded below
NIRCAM_CONF_VERSION = 'V9'        # config version from https://github.com/npirzkal/GRISM_NIRCAM


# Stacking parameters
STACK_TARGETS = True              # after the groups: one combined beams file + one drizzled 2D stack per target
STACK_SEPARATE_DISPERSION = False # False: GRISMR and GRISMC (and all groups) of a target go into the same files
                                  # True : separate files per dispersion direction (<target>_GRISMC.*, <target>_GRISMR.*)
STACK_FLAMBDA = 1e17              # f_lambda units, multiplied by this number: 1e17 -> 1e-17 erg/s/cm2/A (grizli's usual unit).
                                  # Use a NUMBER, not True: with the NIRCam sensitivities True overflows float32 inside grizli, which
                                  # turns the middle of every spectrum into NaN. False -> detector units (e-/s), no conversion.
STACK_1D_METHOD    = 'optimal'    # 'optimal' (profile-weighted, Horne 1986) or 'boxcar' (plain sum of 2*STACK_1D_HALFWIDTH+1 rows)
STACK_1D_HALFWIDTH = 4            # boxcar only: rows on either side of the trace that are summed
STACK_OPT_HALFWIDTH = 8           # optimal only: rows on either side of the trace that the profile is defined on
THUMB_SIZE_ARCSEC  = 3.0          # size of the direct-image cutout stored in the stack and shown in the figure
STACK_SIZE    = 20                # half-height of the drizzled 2D spectra in native pixels (centred on the trace)
STACK_PIXFRAC = 0.5               # drizzle parameters (pixfrac only matters for kernel='point')
STACK_KERNEL  = 'square'
STACK_SKIP_UP_TO_DATE = True      # do not redo a target whose stack is newer than all of its input beams files
SAVE_STACK_PLOTS = True


SKIP_COMPLETED  = True            # skip groups already done for the same targets and settings
SAVE_QC_PLOTS   = True
MAKE_BEAM_PLOTS = True            # one PNG per target and group
ONLY_GROUPS     = None            # None = all, or substrings of group ids


# Paths (should be in ./jwst_wfss/Prep - relative to this notebooks location)
WORKDIR     = Path('./jwst_wfss').resolve()       
PREP_DIR    = WORKDIR / 'Prep'
MANIFEST    = WORKDIR / 'manifest.json'
TARGET_OUT  = WORKDIR / 'TargetExtractions'
STACK_OUT   = WORKDIR / 'Stacked'
for d in (TARGET_OUT, STACK_OUT):
    d.mkdir(parents=True, exist_ok=True)

# Setup CRDS
os.environ['NIRCAM_CONF_VERSION'] = NIRCAM_CONF_VERSION     
os.environ.setdefault('CRDS_PATH', str(Path.home() / 'crds_cache'))
os.environ.setdefault('CRDS_SERVER_URL', 'https://jwst-crds.stsci.edu')

# Import grizli
import grizli
from grizli import utils, prep, multifit, jwst_utils, grismconf

## 1. Read the target catalog

In [ ]:
t = Table.read(TARGET_CATALOG, format='ascii.csv')
names = t[NAME_COL]
ra, dec = t[RA_COL], t[DEC_COL]
tgt = Table([names, ra, dec], names=['name', 'ra', 'dec'])
tgt = tgt[:10]

print(len(tgt), 'targets')
tgt[:10].pprint_all()

## 2. Identify groups that cover targets
### This required the manifest file from Notebook 01

In [8]:
if not MANIFEST.exists():
    raise FileNotFoundError(f'{MANIFEST} not found: run notebook 01 first (and check that WORKDIR is the same).')
manifest = json.loads(MANIFEST.read_text())
settings = manifest['settings']
PIXSCALE = float(settings['PIXSCALE'])          # only used if a mosaic has to be rebuilt
groups_all = manifest['groups']
print(f'manifest: {len(groups_all)} groups (created {manifest["created"]}, grizli {manifest["grizli_version"]})')
if settings.get('NIRCAM_CONF_VERSION') != NIRCAM_CONF_VERSION:
    print(f'NOTE: notebook 01 used NIRCAM_CONF_VERSION={settings.get("NIRCAM_CONF_VERSION")}, this notebook uses {NIRCAM_CONF_VERSION}')
if any(abs(float(r.get('pixscale', PIXSCALE)) - PIXSCALE) > 1e-3 for r in groups_all if r['status'] == 'ok'):
    print('WARNING: mosaics in the manifest have different pixel scales (different runs of notebook 01 with other PIXSCALE).')


def sep_arcsec(ra, dec, ra0, dec0):
    dra = (np.asarray(ra) - ra0 + 180.) % 360. - 180.
    return np.hypot(dra * np.cos(np.deg2rad(dec0)), np.asarray(dec) - dec0) * 3600.


def nearest_target_arcsec(g):
    return float(np.min(sep_arcsec(tgt['ra'], tgt['dec'], g['center_ra'], g['center_dec'])))


ok_groups = [g for g in groups_all if g['status'] == 'ok']
for g in ok_groups:
    g['nearest_arcsec'] = nearest_target_arcsec(g)
near = [g for g in ok_groups if g['nearest_arcsec'] < g['radius_arcsec'] + COARSE_MARGIN_ARCSEC]

# every file the group needs must still be there
groups, missing = [], []
for g in near:
    need = [PREP_DIR / f for f in g['grism_files'] + g['direct_files']] + \
           [PREP_DIR / g[k] for k in ('mosaic_sci', 'mosaic_wht', 'catalog', 'segmentation')]
    lost = [p.name for p in need if not p.exists()]
    (missing if lost else groups).append((g, lost) if lost else g)
groups = [g for g in groups]

print(f'\n{len(ok_groups)} usable groups in the manifest | {len(near)} cover or nearly cover a target | {len(groups)} with all files present')
for g, lost in missing:
    print('  files missing for', g['id'], '->', lost[:3], '...' if len(lost) > 3 else '')
for r in groups_all:
    if r['status'] != 'ok':
        print(f'  not usable: {r["id"]} [{r["status"]}] {r["note"]}')

if not groups:
    print('\nAll groups:')
    Table(rows=[(r['id'], r['status'], np.round(r.get('center_ra', np.nan), 4), np.round(r.get('center_dec', np.nan), 4),
                 np.round(r.get('radius_arcsec', np.nan)), np.round(r.get('nearest_arcsec', np.nan)))
                for r in groups_all],
          names=['group', 'status', 'center_ra', 'center_dec', 'radius_arcsec', 'nearest_target_arcsec']).pprint_all(max_width=200)
    print('\nTargets:'); tgt[:5].pprint_all()
    raise RuntimeError('No group mosaic covers any target. Compare the target coordinates with the table above and the footprint plot '
                       'of notebook 01. If the targets lie in another field, run notebook 01 for it (it merges into the manifest).')

# the grism exposures must have their original PUPIL back (notebook 01 does this)
bad = [(f, fits.getheader(PREP_DIR / f, 0).get('PUPIL')) for f in sorted({f for g in groups for f in g['grism_files']})]
bad = [(f, p) for f, p in bad if not str(p).startswith('GRISM')]
if bad:
    raise RuntimeError(f'{len(bad)} grism exposures have PUPIL != GRISM*, e.g. {bad[0]}: re-run the initialisation cell of notebook 01 '
                       '(it restores the keywords).')

Table(rows=[(g['id'], len(g['grism_files']), len(g['direct_files']), g['direct_filter'], round(g['nearest_arcsec']), round(g['radius_arcsec']))
            for g in groups],
      names=['group', 'N_grism', 'N_direct', 'direct_filter', 'nearest_target_arcsec', 'mosaic_radius_arcsec']).pprint_all(max_width=200)

manifest: 30 groups (created 2026-11-15T19:40:54, grizli 1.14.3)

28 usable groups in the manifest | 28 cover or nearly cover a target | 28 with all files present
  not usable: A2744-jw03538007001-modA-f410m-grismc [skipped] no A-module direct image in F410M found
  not usable: A2744-jw03538007001-modB-f410m-grismc [skipped] no B-module direct image in F410M found
                group                 N_grism N_direct direct_filter nearest_target_arcsec mosaic_radius_arcsec
------------------------------------- ------- -------- ------------- --------------------- --------------------
A2744-jw03538001001-modA-f300m-grismc       2        1         F300M                    41                  127
A2744-jw03538001001-modB-f300m-grismc       2        1         F300M                    78                  127
A2744-jw03538001004-modA-f300m-grismc       2        1         F300M                    30                  127
A2744-jw03538001004-modB-f300m-grismc       2        1         F300M     

## 3. Prepare mosaics and config files

In [ ]:
import sep
from scipy import ndimage as nd

BAD_DQ = 1 | 2 | 1024 | 2048 | 8192 | 16384   # DO_NOT_USE, SATURATED, DEAD, HOT, LOW_QE, RC

# Helper function to load a header keyword
def get_key(files, key, default=None):
    for f in files:
        with fits.open(f) as hdul:
            for ext in (0, 'SCI'):
                if key in hdul[ext].header:
                    return hdul[ext].header[key]
    return default


# Same as in Notebook 01 - used to build mosaics if not previously saved
def build_mosaic(root, direct_files):
    sci_file, wht_file = f'{root}_drz_sci.fits', f'{root}_drz_wht.fits'
    if os.path.exists(sci_file) and os.path.exists(wht_file):
        return sci_file, wht_file

    sci_list, wht_list, wcs_list = [], [], []
    for f in direct_files:
        with fits.open(f) as hdul:
            sci = np.ascontiguousarray(hdul['SCI'].data, dtype=np.float32)
            err = np.ascontiguousarray(hdul['ERR'].data, dtype=np.float32)
            dq = hdul['DQ'].data
            w = pywcs.WCS(hdul['SCI'].header, fobj=hdul, relax=True)
        w.pixel_shape = sci.shape[::-1]
        w.pscale = utils.get_wcs_pscale(w)

        bad = ((dq & BAD_DQ) > 0) | ~np.isfinite(sci) | ~np.isfinite(err) | (err <= 0)
        sci[bad] = 0
        bkg = sep.Background(sci, mask=bad, bw=64, bh=64, fw=3, fh=3)
        sci = (sci - bkg.back()).astype(np.float32)
        wht = np.where(bad, 0, 1.0 / np.maximum(err, 1e-12) ** 2).astype(np.float32)
        sci[bad] = 0
        sci_list.append(sci); wht_list.append(wht); wcs_list.append(w)

    fp = np.vstack([w.calc_footprint() for w in wcs_list])
    ra = ((fp[:, 0] - fp[0, 0] + 180) % 360) - 180 + fp[0, 0]
    dec = fp[:, 1]
    ra0, dec0 = 0.5 * (ra.min() + ra.max()), 0.5 * (dec.min() + dec.max())

    tmp = pywcs.WCS(naxis=2)
    tmp.wcs.ctype = ['RA---TAN', 'DEC--TAN']
    tmp.wcs.crval = [ra0, dec0]
    tmp.wcs.cdelt = [-PIXSCALE / 3600., PIXSCALE / 3600.]
    tmp.wcs.crpix = [1., 1.]
    x, y = tmp.wcs_world2pix(ra, dec, 1)
    pad = 20
    nx = int(np.ceil(x.max() - x.min())) + 2 * pad
    ny = int(np.ceil(y.max() - y.min())) + 2 * pad

    wcs_hdr = fits.Header()
    wcs_hdr['NAXIS'] = 2
    wcs_hdr['NAXIS1'], wcs_hdr['NAXIS2'] = nx, ny
    wcs_hdr['CTYPE1'], wcs_hdr['CTYPE2'] = 'RA---TAN', 'DEC--TAN'
    wcs_hdr['CUNIT1'], wcs_hdr['CUNIT2'] = 'deg', 'deg'
    wcs_hdr['CRPIX1'] = 1. - x.min() + pad
    wcs_hdr['CRPIX2'] = 1. - y.min() + pad
    wcs_hdr['CRVAL1'], wcs_hdr['CRVAL2'] = ra0, dec0
    wcs_hdr['CD1_1'], wcs_hdr['CD1_2'] = -PIXSCALE / 3600., 0.
    wcs_hdr['CD2_1'], wcs_hdr['CD2_2'] = 0., PIXSCALE / 3600.
    wcs_hdr['RADESYS'] = 'ICRS'

    out_wcs = pywcs.WCS(wcs_hdr)
    out_wcs.pixel_shape = (nx, ny)
    out_wcs.pscale = utils.get_wcs_pscale(out_wcs)

    res = utils.drizzle_array_groups(sci_list, wht_list, wcs_list, outputwcs=out_wcs, scale=PIXSCALE,
                                     kernel='square', pixfrac=0.8, verbose=False)
    outsci, outwht = res[0], res[1]

    hdr = wcs_hdr.copy()
    for k in ['TELESCOP', 'INSTRUME', 'DETECTOR', 'FILTER', 'PUPIL', 'MODULE',
              'OTELESCOP', 'OINSTRUME', 'ODETECTO', 'OFILTER', 'OPUPIL', 'OEXP_TYP',
              'PHOTFLAM', 'PHOTPLAM', 'PHOTMJSR', 'PHOTUJA2']:
        v = get_key(direct_files, k)
        if v is not None:
            hdr[k] = v
    hdr['EXPTIME'] = float(sum(get_key([f], 'EFFEXPTM', 0.) for f in direct_files))
    hdr['BUNIT'] = 'DN/s'
    for k in ('PHOTFLAM', 'PHOTPLAM'):
        if k not in hdr:
            raise RuntimeError(f'{k} missing from the mosaic header (direct images not initialised?)')

    fits.writeto(sci_file, outsci.astype(np.float32), header=hdr, overwrite=True)
    fits.writeto(wht_file, outwht.astype(np.float32), header=hdr, overwrite=True)
    print(f'  mosaic {root}: {nx} x {ny} px from {len(direct_files)} direct exposure(s)')
    return sci_file, wht_file

# Same as Notebook 01
def build_catalog(root, sci_file, wht_file):
    cat_file, seg_file = f'{root}.cat.fits', f'{root}_seg.fits'
    if not (os.path.exists(cat_file) and os.path.exists(seg_file)):
        prep.make_SEP_catalog(root=root, sci=sci_file, wht=wht_file, threshold=2.0, get_background=True)
    for f in (cat_file, seg_file):
        if not os.path.exists(f):
            raise RuntimeError(f'{f} not written by make_SEP_catalog: {glob.glob(root + "*")}')
    return cat_file, seg_file


# Get the image cutout
def required_cutout(flt, margin=CUTOUT_MARGIN, lo=26, hi=200, default=80):
    conf = flt.conf
    dmax = 0.0
    for (x, y) in [(200, 200), (1024, 1024), (1800, 1800), (200, 1800), (1800, 200)]:
        for beam in conf.beams:
            dy, lam = conf.get_beam_trace(x=x, y=y, dx=conf.dxlam[beam], beam=beam)
            dmax = max(dmax, float(np.nanmax(np.abs(dy))))
    return int(np.clip(np.ceil(dmax + margin), lo, hi)), dmax
  

# Determine the model for the flt (using grizlis tools)
def compute_model(grp, cutout):
    cat_g = grp.catalog
    idcol = 'NUMBER' if 'NUMBER' in cat_g.colnames else 'id'
    mag = np.asarray(cat_g['MAG_AUTO'], dtype=float)
    sel = np.isfinite(mag) & (mag < MAG_LIMIT_MODEL)
    ids = np.asarray(cat_g[idcol])[sel].astype(int)
    mags = mag[sel]
    for flt in grp.FLTs:
        flt.model[:] = 0
        flt.object_dispersers = OrderedDict()
        flt.compute_full_model(ids=ids, mags=mags, mag_limit=MAG_LIMIT_MODEL, store=False,
                               verbose=False, min_size=cutout, compute_size=True)
    n_model = [len(flt.object_dispersers) for flt in grp.FLTs]
    mmax = [float(flt.model.max()) for flt in grp.FLTs]
    return len(ids), n_model, mmax


## 4. Locate targets in images

In [ ]:
from astropy.wcs.utils import proj_plane_pixel_scales

# Helper functions for pixel scales and magnitude conversions
def mosaic_pixel_scale(w):
    return float(np.mean(proj_plane_pixel_scales(w))) * 3600.

def ab_zeropoint(hdr):
    return -2.5 * np.log10(hdr['PHOTFLAM']) - 21.10 - 5 * np.log10(hdr['PHOTPLAM']) + 18.6921

# Identify targets that fall on each direct image
def locate_targets(g):
    sci_file, wht_file = build_mosaic(g['root'], g['direct_files'])
    w = pywcs.WCS(fits.getheader(sci_file))
    wht = fits.getdata(wht_file)
    ny, nx = wht.shape
    x, y = w.all_world2pix(np.asarray(tgt['ra'], float), np.asarray(tgt['dec'], float), 0)
    hits = []
    for i in range(len(tgt)):
        xi, yi = int(round(float(x[i]))), int(round(float(y[i])))
        if 0 <= xi < nx and 0 <= yi < ny and wht[yi, xi] > 0:
            hits.append(dict(i=i, name=str(tgt['name'][i]), ra=float(tgt['ra'][i]),
                             dec=float(tgt['dec'][i]), x=float(x[i]), y=float(y[i]),
                             id=None, kind='', sep_arcsec=np.nan, mag=np.nan))
    return hits


# Assign to segmentation image, if not, run add_new_sources
def match_to_segmentation(hits, seg, pscale):
    r_pix = max(MATCH_RADIUS_ARCSEC / pscale, 1.0)
    ny, nx = seg.shape
    for h in hits:
        xlo, xhi = int(max(0, np.floor(h['x'] - r_pix))), int(min(nx, np.ceil(h['x'] + r_pix) + 1))
        ylo, yhi = int(max(0, np.floor(h['y'] - r_pix))), int(min(ny, np.ceil(h['y'] + r_pix) + 1))
        sub = seg[ylo:yhi, xlo:xhi]
        yy, xx = np.mgrid[ylo:yhi, xlo:xhi]
        dist = np.hypot(xx - h['x'], yy - h['y'])
        ok = (sub > 0) & (dist <= r_pix)
        if ok.any():
            k = int(np.argmin(np.where(ok, dist, np.inf)))
            h['id'] = int(sub.flat[k])
            h['sep_arcsec'] = float(dist.flat[k] * pscale)
            h['kind'] = 'detected'
    return hits

# Inject a point in the segmentation image to exrtact from
def add_new_sources(g, hits, cat_file, seg_file, sci_file, pscale):
    todo = [h for h in hits if h['id'] is None]
    if not todo:
        return cat_file, seg_file
    cat = Table.read(cat_file)
    if 'NUMBER' not in cat.colnames or 'MAG_AUTO' not in cat.colnames:
        raise RuntimeError(f'catalog {cat_file} lacks NUMBER / MAG_AUTO columns: {cat.colnames}')
    with fits.open(seg_file) as hdul:
        seg = np.array(hdul[0].data, dtype=np.int32)
        seg_hdr = hdul[0].header.copy()
    sci = fits.getdata(sci_file)
    zp = ab_zeropoint(fits.getheader(sci_file))
    ny, nx = seg.shape

    next_id = int(max(int(np.max(cat['NUMBER'])), int(seg.max()))) + 1
    r_pix = max(NEW_SOURCE_RADIUS_ARCSEC / pscale, 1.5)
    added = []
    for h in todo:
        xlo, xhi = int(max(0, np.floor(h['x'] - r_pix - 1))), int(min(nx, np.ceil(h['x'] + r_pix) + 2))
        ylo, yhi = int(max(0, np.floor(h['y'] - r_pix - 1))), int(min(ny, np.ceil(h['y'] + r_pix) + 2))
        yy, xx = np.mgrid[ylo:yhi, xlo:xhi]
        circle = np.hypot(xx - h['x'], yy - h['y']) <= r_pix
        sub = seg[ylo:yhi, xlo:xhi]
        free = circle & (sub == 0)
        if free.sum() == 0:
            h['kind'] = 'skipped (no free segmentation pixels)'
            continue
        sub[free] = next_id                              
        flux = float(np.nansum(sci[ylo:yhi, xlo:xhi][circle]))
        mag = float(zp - 2.5 * np.log10(flux)) if flux > 0 else 99.0
        h.update(id=next_id, kind='new segment', sep_arcsec=0.0, mag=mag)
        added.append((next_id, h, mag, flux))
        next_id += 1

    if added:
        new = Table(np.zeros(len(added), dtype=cat.dtype))
        def setcol(names, values):
            for n in names:
                if n in new.colnames:
                    new[n] = values
        setcol(['NUMBER'], [a[0] for a in added])
        setcol(['X_WORLD', 'ra', 'RA'], [a[1]['ra'] for a in added])
        setcol(['Y_WORLD', 'dec', 'DEC'], [a[1]['dec'] for a in added])
        setcol(['X_IMAGE'], [a[1]['x'] + 1 for a in added])
        setcol(['Y_IMAGE'], [a[1]['y'] + 1 for a in added])
        setcol(['x'], [a[1]['x'] for a in added])
        setcol(['y'], [a[1]['y'] for a in added])
        setcol(['MAG_AUTO'], [a[2] for a in added])
        setcol(['FLUX_AUTO'], [a[3] for a in added])
        setcol(['A_IMAGE', 'B_IMAGE'], [r_pix] * len(added))
        cat = vstack([cat, new], metadata_conflicts='silent')

    cat_out, seg_out = f'{g["id"]}_targets.cat.fits', f'{g["id"]}_targets_seg.fits'
    cat.write(cat_out, overwrite=True)
    fits.writeto(seg_out, seg, header=seg_hdr, overwrite=True)
    print(f'  added {len(added)} new segment(s) for undetected targets -> {cat_out}')
    return cat_out, seg_out

# Helper code to plot beams
def plot_beams(mb, path=None, vmax_pct=99):
    n = len(mb.beams)
    fig, axes = plt.subplots(n, 4, figsize=(16, 1.6 * n + 0.8), squeeze=False)
    for i, b in enumerate(mb.beams):
        sci, mod, con = b.grism['SCI'], b.beam.model, b.contam
        vmax = np.nanpercentile(sci, vmax_pct) if np.isfinite(sci).any() else 1
        for ax, im, t in zip(axes[i], [sci, mod, con, sci - con - mod], ['data', 'model', 'contam', 'data-contam-model']):
            ax.imshow(im, origin='lower', aspect='auto', cmap='gray_r', vmin=-0.1 * abs(vmax), vmax=max(vmax, 1e-6))
            ax.set_xticks([]); ax.set_yticks([])
            if i == 0:
                ax.set_title(t)
        axes[i, 0].set_ylabel(f'{b.grism.parent_file[:18]}', fontsize=6)
    fig.suptitle(f'{mb.group_name}  (grizli ID {mb.id})')
    plt.tight_layout()
    if path:
        fig.savefig(path, dpi=80)
        plt.close(fig)
    else:
        plt.show()

In [ ]:
RESULT_COLS = ['target', 'ra', 'dec', 'group', 'match', 'grizli_id', 'sep_arcsec', 'mag', 'n_beams', 'beams_file', 'status']

### The main extraction function
def process_group_targets(g, hits):
    t0 = time.time()
    out_dir = TARGET_OUT / g['id']
    out_dir.mkdir(parents=True, exist_ok=True)
    rows, grp, bkg_levels = [], None, []
    try:
        os.chdir(PREP_DIR)
        sci_file, wht_file = build_mosaic(g['root'], g['direct_files'])
        cat_file, seg_file = build_catalog(g['root'], sci_file, wht_file)
        pscale = mosaic_pixel_scale(pywcs.WCS(fits.getheader(sci_file)))

        match_to_segmentation(hits, fits.getdata(seg_file), pscale)
        cat_use, seg_use = add_new_sources(g, hits, cat_file, seg_file, sci_file, pscale)
        n_det = sum(h['kind'] == 'detected' for h in hits)
        print(f'  {len(hits)} targets on this mosaic: {n_det} matched to detections, '
              f'{sum(h["kind"] == "new segment" for h in hits)} new segments')

        usable = [h for h in hits if h['id'] is not None]
        if not usable:
            raise RuntimeError('no target could be assigned a segment')

        grp = multifit.GroupFLT(grism_files=g['grism_files'], direct_files=[], ref_file=sci_file,
                                seg_file=seg_use, catalog=cat_use, cpu_count=CPU_COUNT, sci_extn=1,
                                pad=PAD, use_jwst_crds=USE_JWST_CRDS)

        for flt in grp.FLTs:                     
            letter = str(flt.grism.pupil)[-1]
            cf = os.path.basename(str(flt.conf_file))
            if not cf.endswith(f'_{letter}.conf'):
                raise RuntimeError(f'config {cf} does not match pupil {flt.grism.pupil}: header restore failed?')
        conf_name = os.path.basename(str(grp.FLTs[0].conf_file))

        cutout, dmax = required_cutout(grp.FLTs[0])
        print(f'  config {conf_name} | max trace offset {dmax:.0f} px -> cutout half-size {cutout} px')
        n_cat, n_model, mmax = compute_model(grp, cutout)
        if min(mmax) == 0:
            raise RuntimeError('empty contamination model in at least one exposure')
        print(f'  model: {n_cat} sources < mag {MAG_LIMIT_MODEL}, objects per exposure {n_model}')
        for h in hits:
            row = dict(target=h['name'], ra=h['ra'], dec=h['dec'], group=g['id'], match=h['kind'],
                       grizli_id=h['id'] if h['id'] is not None else -1,
                       sep_arcsec=round(h['sep_arcsec'], 3) if np.isfinite(h['sep_arcsec']) else np.nan,
                       mag=round(h['mag'], 2) if np.isfinite(h['mag']) else np.nan,
                       n_beams=0, beams_file='', status='')
            if h['id'] is None:
                row['status'] = 'no segment'
                rows.append(row); continue
            try:
                beams = grp.get_beams(int(h['id']), size=cutout, beam_id='A')
                if len(beams) == 0:
                    row['status'] = 'no grism overlap'
                else:
                    mb = multifit.MultiBeam(beams, fcontam=FCONTAM, group_name=h['safe'])
                    mb.write_master_fits()
                    row.update(n_beams=len(beams), beams_file=f'{h["safe"]}_{int(h["id"]):05d}.beams.fits', status='OK')
                    if MAKE_BEAM_PLOTS:
                        plot_beams(mb, path=out_dir / f'{h["safe"]}.png')
            except Exception as e:
                row['status'] = 'FAILED: ' + repr(e)[:100]
            rows.append(row)
            print(f'   {h["name"]:20s} {row["match"]:14s} id={row["grizli_id"]:<7d} {row["status"]}')
    finally:
        os.chdir(PREP_DIR)
        if grp is not None:
            del grp
        gc.collect()
    print(f'  group done in {(time.time() - t0) / 60:.1f} min')
    return rows

## 5. Extract spectra

In [ ]:
def run_key(hits):
    items = sorted((h['name'], round(h['ra'], 7), round(h['dec'], 7)) for h in hits)
    return hashlib.md5(repr((items, MATCH_RADIUS_ARCSEC, NEW_SOURCE_RADIUS_ARCSEC, ADD_UNDETECTED,
                             MAG_LIMIT_MODEL, FCONTAM, CUTOUT_MARGIN,)).encode()).hexdigest()


all_rows, hits_by_group = [], {}
run_groups = [g for g in groups if not ONLY_GROUPS or any(s in g['id'] for s in ONLY_GROUPS)]

for gi, g in enumerate(run_groups, 1):
    print(f'\n===== [{gi}/{len(run_groups)}] {g["id"]} =====')

    os.chdir(PREP_DIR)
    hits = locate_targets(g)
    if not hits:
        print('  no targets on this mosaic')
        continue
    hits_by_group[g['id']] = hits

    out_dir = TARGET_OUT / g['id']
    marker, rfile = out_dir / '.done', out_dir / 'results.csv'
    key = run_key(hits)
    if SKIP_COMPLETED and marker.exists() and rfile.exists() and marker.read_text().strip() == key:
        print('  already done for these targets, loading earlier results')
        all_rows += [dict(zip(RESULT_COLS, r)) for r in Table.read(rfile, format='ascii.csv').as_array().tolist()]
        continue

    rows = process_group_targets(g, hits)
    out_dir.mkdir(parents=True, exist_ok=True)
    Table(rows=[[r.get(c, '') for c in RESULT_COLS] for r in rows], names=RESULT_COLS
            ).write(rfile, format='ascii.csv', overwrite=True)
    marker.write_text(key)
    all_rows += rows

    os.chdir(PREP_DIR)
    plt.close('all')
    gc.collect()


results = Table(rows=[[r.get(c, '') for c in RESULT_COLS] for r in all_rows], names=RESULT_COLS)
results.write(WORKDIR / 'target_results.csv', format='ascii.csv', overwrite=True)


### 6. Stack all extractions of each target

In [ ]:
from astropy.nddata import Cutout2D
from astropy.coordinates import SkyCoord
from scipy import ndimage as ndi


FLAM_SCALE = 1e17 if STACK_FLAMBDA is True else STACK_FLAMBDA  
if FLAM_SCALE and FLAM_SCALE == 1e17:
    FLAM_LABEL = r'$f_\lambda$ [$10^{-17}$ erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$]'
elif FLAM_SCALE:
    FLAM_LABEL = rf'$f_\lambda$ x {FLAM_SCALE:g} [erg s$^{{-1}}$ cm$^{{-2}}$ $\AA^{{-1}}$]'
else:
    FLAM_LABEL = 'counts [e-/s] (no flux calibration)'


# Retrieve the object cutout from the direct image
def direct_thumbnail(rows, size_arcsec):
    group_info = {g['id']: g for g in groups}
    for r in rows:
        g = group_info.get(str(r['group']))
        if g is None:
            continue
        f = PREP_DIR / g['mosaic_sci']
        if not f.exists():
            continue
        try:
            with fits.open(f, memmap=True) as hdul:
                w = pywcs.WCS(hdul[0].header)
                pos = SkyCoord(float(r['ra']) * u.deg, float(r['dec']) * u.deg)
                cut = Cutout2D(hdul[0].data, pos, size_arcsec * u.arcsec, wcs=w, mode='partial', fill_value=np.nan)
                hdr = cut.wcs.to_header()
                for key in ('BUNIT', 'PHOTFLAM', 'PHOTPLAM', 'FILTER'):
                    if key in hdul[0].header:
                        hdr[key] = hdul[0].header[key]
                hdr['GROUP'] = (str(r['group'])[:60], 'Group the cutout comes from')
                return fits.ImageHDU(np.asarray(cut.data, dtype=np.float32), header=hdr, name='THUMB')
        except Exception as e:
            print(f'   thumbnail from {f.name} failed: {repr(e)[:100]}')
    return None



# 1d exrtaction
def extract_1d(sci, wht, mod, header):
    d, wt = np.asarray(sci, float), np.asarray(wht, float)
    ny, nx = d.shape
    wave = header['CRVAL1'] + (np.arange(nx) + 1 - header['CRPIX1']) * header['CD1_1']
    yc = int(round(header['CRPIX2'] - 1))
    prof = None if mod is None else np.clip(np.nan_to_num(np.asarray(mod, float)), 0, None)
    r0, r1 = max(yc - STACK_OPT_HALFWIDTH, 0), min(yc + STACK_OPT_HALFWIDTH + 1, ny)
    P = ndi.uniform_filter1d(prof[r0:r1], size=25, axis=1, mode='nearest')
    tot = P.sum(axis=0)
    D, W = d[r0:r1], np.where(wt[r0:r1] > 0, wt[r0:r1], 0.0)
    with np.errstate(divide='ignore', invalid='ignore'):
        Pn = np.where(tot > 0, P / tot, 0.0)
        cover = (Pn * (W > 0)).sum(axis=0)
        den = (Pn ** 2 * W).sum(axis=0)
        flux = (Pn * D * W).sum(axis=0) / den
        err = 1.0 / np.sqrt(den)
    ok = (tot > 0) & (cover >= 0.9) & (den > 0)
    return wave, np.where(ok, flux, np.nan), np.where(ok, err, np.nan), 'optimal'


# Produce 1D spectrum fits information
def spec1d_hdus(sh):
    out = []
    for e in sh:
        if e.name != 'SCI' or ',' in str(e.header.get('EXTVER', ',')):
            continue
        flt = e.header['EXTVER']
        mod = sh['MODEL', flt].data if ('MODEL', flt) in sh else None
        wave, flux, err, used = extract_1d(e.data, sh['WHT', flt].data, mod, e.header)
        h = fits.table_to_hdu(Table([wave, flux, err], names=['wave', 'flux', 'err']))
        h.name, h.header['EXTVER'] = 'SPEC1D', flt
        h.header['METHOD'] = (used, '1D extraction method')
        h.header['FLAMSCL'] = (float(FLAM_SCALE) if FLAM_SCALE else 0.0, 'f_lambda multiplier (0 = e-/s)')
        out.append(h)
    return out

# Stack the full spectrum
def stack_figure(sh, thumb, title, path=None, halfwidth=None):
    halfwidth = STACK_1D_HALFWIDTH if halfwidth is None else halfwidth
    filters = [e.header['EXTVER'] for e in sh if e.name == 'SCI' and ',' not in str(e.header.get('EXTVER', ','))]
    if not filters:
        return None
    from matplotlib.gridspec import GridSpec
    n = len(filters)
    fig = plt.figure(figsize=(12, 3.6 * n))
    gs = GridSpec(2 * n, 2, width_ratios=[1, 4.5], height_ratios=[1.2, 1.0] * n, hspace=0.12, wspace=0.3)
    for j, flt in enumerate(filters):
        sci, wht = sh['SCI', flt], sh['WHT', flt]
        d, wt, hd = np.asarray(sci.data, float), np.asarray(wht.data, float), sci.header
        ny, nx = d.shape
        wave = hd['CRVAL1'] + (np.arange(nx) + 1 - hd['CRPIX1']) * hd['CD1_1']
        dw = hd['CD1_1']
        covered = (wt > 0).any(axis=0)
        # direct image
        axt = fig.add_subplot(gs[2 * j:2 * j + 2, 0])
        if thumb is not None and np.isfinite(thumb.data).any():
            t = thumb.data
            lo, hi = np.nanpercentile(t, [1, 99.7])
            axt.imshow(t, origin='lower', cmap='gray_r', vmin=lo, vmax=max(hi, lo + 1e-9))
            axt.set_title(f'direct image\n{THUMB_SIZE_ARCSEC:g}" cutout (N up, E left)', fontsize=8)
        else:
            axt.text(0.5, 0.5, 'no direct\nimage', ha='center', va='center', transform=axt.transAxes)
        axt.set_xticks([]); axt.set_yticks([])
        # 2D spectrum
        ax2 = fig.add_subplot(gs[2 * j, 1])
        good = wt > 0
        vmax = np.nanpercentile(d[good], 99) if good.any() else 1.
        ax2.imshow(np.where(good, d, np.nan), origin='lower', aspect='auto', cmap='viridis_r', vmin=-0.1 * abs(vmax), vmax=vmax,
                   extent=[wave[0] - dw / 2, wave[-1] + dw / 2, -hd['CRPIX2'] + 0.5, ny - hd['CRPIX2'] + 0.5])
        ax2.set_ylabel('spatial [px]')
        ax2.set_title(f'{title} | {flt} | {sh[0].header.get("NINPUT", "?")} input group(s)', fontsize=9)
        ax2.tick_params(labelbottom=False)
        # 1D spectrum
        ax1 = fig.add_subplot(gs[2 * j + 1, 1], sharex=ax2)
        mod = sh['MODEL', flt].data if ('MODEL', flt) in sh else None
        _, flux, err, used = extract_1d(d, wt, mod, hd)
        ax1.fill_between(wave, flux - err, flux + err, color='0.8', step='mid')
        ax1.step(wave, flux, where='mid', color='k', lw=0.8)
        ax1.axhline(0, color='r', lw=0.5)
        ax1.set_ylabel(FLAM_LABEL, fontsize=8)
        ax1.text(0.99, 0.95, used + ' extraction', transform=ax1.transAxes, ha='right', va='top', fontsize=7, color='0.4')
        ax1.set_xlabel(r'$\lambda$ [$\mu$m]')
        if covered.any():
            ax2.set_xlim(wave[covered].min() - dw, wave[covered].max() + dw)
        fin = flux[np.isfinite(flux)]
        if fin.size:
            lo, hi = np.percentile(fin, [1, 99])
            pad = 0.2 * (hi - lo + 1e-30)
            ax1.set_ylim(lo - pad, hi + 3 * pad)
    if path is not None:
        fig.savefig(path, dpi=90, bbox_inches='tight')
    return fig


STACK_OUT.mkdir(parents=True, exist_ok=True)
try:
    results
except NameError:       # fresh kernel: use the table written by the run cell
    results = Table.read(WORKDIR / 'target_results.csv', format='ascii.csv')

group_pupils = {g['id']: '+'.join(g['pupils']) for g in groups}

def dispersion_key(r):
    return group_pupils.get(str(r['group']), '') if STACK_SEPARATE_DISPERSION else ''

def stack_one(name, key, rows):
    safe = name
    base = STACK_OUT / (safe + (('_' + key.replace('+', '-')) if key else ''))
    beams_f, stack_f = Path(f'{base}.beams.fits'), Path(f'{base}.stack.fits')
    files = [str(TARGET_OUT / str(r['group']) / str(r['beams_file'])) for r in rows]
    out = dict(target=name, dispersion=key, n_inputs=len(files), n_beams=0, grisms='',
                beams_file=beams_f.name, stack_file=stack_f.name, status='')
    missing = [f for f in files if not os.path.exists(f)]
    if missing:
        raise FileNotFoundError(f'{len(missing)} input beams files missing, e.g. {missing[0]}')
    newest = max(os.path.getmtime(f) for f in files)
    if STACK_SKIP_UP_TO_DATE and beams_f.exists() and stack_f.exists() and \
            min(os.path.getmtime(beams_f), os.path.getmtime(stack_f)) > newest:
        out['status'] = 'up to date'
        return out

    mb = multifit.MultiBeam(files, fcontam=FCONTAM, group_name=safe, verbose=False)
    out['n_beams'] = int(mb.N)
    out['grisms'] = ', '.join(f'{g}:{n}' for g, n in mb.Ngrism.items())

    # (1) every beam of the target in one master file
    hdu = mb.write_master_fits(get_hdu=True, verbose=False)
    hdu[0].header['TARGET'] = (name[:60], 'Target name')
    hdu[0].header['NINPUT'] = (len(files), 'Input beams files (groups); ID is that of the first')
    hdu.writeto(beams_f, overwrite=True)

    # (2) drizzled 2D stacks per crossing filter and per position angle
    known = [g for g in mb.PA if g in utils.GRISM_LIMITS]
    unknown = [g for g in mb.PA if g not in utils.GRISM_LIMITS]
    if unknown:
        print(f'   {name}: no wavelength limits in utils.GRISM_LIMITS for {unknown}: not stacked')
    if not known:
        raise RuntimeError(f'no stackable grism among {list(mb.PA)}')
    kw = dict(size=STACK_SIZE, fcontam=FCONTAM, flambda=(FLAM_SCALE if FLAM_SCALE else False), scale=1,
                pixfrac=STACK_PIXFRAC, kernel=STACK_KERNEL, grism_list=known)
    try:
        res = mb.drizzle_grisms_and_PAs(make_figure=False, **kw)
    except TypeError:                      # grizli versions without make_figure
        res = mb.drizzle_grisms_and_PAs(**kw)
    sh = res[0] if isinstance(res, tuple) else res

    # NaN / inf in a stack means the weights overflowed or a beam was empty: say so, and keep the files clean
    bad = []
    for e in sh[1:]:
        if e.data is not None and e.name in ('SCI', 'WHT', 'MODEL', 'KERNEL', 'CONTAM') and not np.isfinite(e.data).all():
            bad.append(f'{e.name},{e.header.get("EXTVER", "")}')
            e.data = np.nan_to_num(e.data, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    if bad:
        print(f'   WARNING {name}: non-finite pixels set to 0 in {len(bad)} extension(s), e.g. {bad[0]}')

    sh[0].header['TARGET'] = (name[:60], 'Target name')
    sh[0].header['NINPUT'] = (len(files), 'Input beams files (groups)')
    sh[0].header['FLAMSCL'] = (float(FLAM_SCALE) if FLAM_SCALE else 0.0, 'f_lambda multiplier (0 = e-/s)')
    sh[0].header['FLAMUNIT'] = ('1e-17 erg/s/cm2/A' if FLAM_SCALE == 1e17 else
                                ('erg/s/cm2/A x FLAMSCL' if FLAM_SCALE else 'e-/s'), 'Unit of SCI/WHT/MODEL')
    thumb = direct_thumbnail(rows, THUMB_SIZE_ARCSEC)
    if thumb is not None:
        sh.append(thumb)
    inputs = Table(rows=[(str(r['group']), str(r['grizli_id']), str(r['match']), str(r['beams_file'])) for r in rows],
                    names=['group', 'grizli_id', 'match', 'beams_file'])
    ih = fits.table_to_hdu(inputs)
    ih.name = 'INPUTS'
    sh.append(ih)
    sh.extend(spec1d_hdus(sh))
    sh.writeto(stack_f, overwrite=True)

    if SAVE_STACK_PLOTS:
        try:
            fig = stack_figure(sh, thumb, name, path=f'{base}.stack.png')
            plt.close(fig)
            fig = multifit.show_drizzle_HDU(sh, diff=True)       # grizli's own view, incl. every position angle
            fig.savefig(f'{base}.drizzle.png', dpi=80)
        except Exception as e:
            print(f'   {name}: stack figure failed: {repr(e)[:100]}')
    out['status'] = 'OK' + (f' ({len(unknown)} grism(s) not stacked)' if unknown else '') + (' [non-finite pixels]' if bad else '')
    return out



by_target = OrderedDict()
print(results)
for r in results:
    if str(r['status']) == 'OK':
        by_target.setdefault((str(r['target']), dispersion_key(r)), []).append(r)

stack_rows = []
t_stack = time.time()
for ti, ((name, key), rows) in enumerate(by_target.items(), 1):
    try:
        out = stack_one(name, key, rows)
    except Exception as e:
        traceback.print_exc()
        out = dict(target=name, dispersion=key, n_inputs=len(rows), n_beams=0, grisms='', beams_file='',
                    stack_file='', status='FAILED: ' + repr(e)[:120])
    finally:
        plt.close('all')
        gc.collect()
    stack_rows.append(out)
    print(f'[{ti}/{len(by_target)}] {name:25s} {key:14s} inputs={out["n_inputs"]} beams={out["n_beams"]} {out["grisms"]} -> {out["status"]}')

cols = ['target', 'dispersion', 'n_inputs', 'n_beams', 'grisms', 'beams_file', 'stack_file', 'status']
print(stack_rows)
if stack_rows:
    stack_table = Table(rows=[[r[c] for c in cols] for r in stack_rows], names=cols)
    stack_table.write(WORKDIR / 'stack_results.csv', format='ascii.csv', overwrite=True)
    print(f'\n{len(stack_rows)} stacks in {(time.time() - t_stack) / 60:.1f} min -> {STACK_OUT}')
else:
    print('No successful extractions to stack.')
